In [1]:
import pandas as pd

train = pd.read_parquet('../data/processed/features_train.parquet')
test = pd.read_parquet('../data/processed/features_test.parquet')


In [2]:
NON_FEATURES = ['target', 'issue_d', 'earliest_cr_line', 'zip_code', 'addr_state']

X_train = train.drop(columns=NON_FEATURES)
y_train = train['target']

X_test  = test.drop(columns=NON_FEATURES)
y_test  = test['target']

In [3]:
print(X_train.shape, X_test.shape)
print("train bad rate:", y_train.mean().round(4))
print("test bad rate: ", y_test.mean().round(4))

(20902, 71) (5378, 71)
train bad rate: 0.2005
test bad rate:  0.2365


In [4]:
import numpy as np

for name, X in [("train", X_train), ("test", X_test)]:
    print(f"--- {name} ---")
    print("shape:        ", X.shape)
    print("missing:      ", X.isna().sum().sum())
    print("inf:          ", np.isinf(X).sum().sum())
    print("constant cols:", X.columns[X.nunique() == 1].tolist())
    print()

--- train ---
shape:         (20902, 71)
missing:       0
inf:           0
constant cols: []

--- test ---
shape:         (5378, 71)
missing:       0
inf:           0
constant cols: []



In [6]:
from sklearn.preprocessing import StandardScaler

# the scaler works by position, so the two halves must line up exactly
assert list(X_train.columns) == list(X_test.columns), "column order differs"

scaler = StandardScaler()
scaler.fit(X_train)          # means and stds learned from train only

X_train_s = pd.DataFrame(
    scaler.transform(X_train),
    columns=X_train.columns,
    index=X_train.index,
)
X_test_s = pd.DataFrame(
    scaler.transform(X_test),   # train's numbers, applied to test
    columns=X_test.columns,
    index=X_test.index,
)

print("train  mean:", round(X_train_s.mean().mean(), 6),
      " std:", round(X_train_s.std().mean(), 4))
print("test   mean:", round(X_test_s.mean().mean(), 4),
      " std:", round(X_test_s.std().mean(), 4))

train  mean: -0.0  std: 1.0
test   mean: -0.0066  std: 1.0813


In [7]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

model = LogisticRegression(
    max_iter=2000,      # high enough that the solver settles; watch for a warning
    random_state=42,    # same run every time
)
model.fit(X_train_s, y_train)   # scaled features, not the originals

# column 1 is P(default); column 0 is P(no default)
p_train = model.predict_proba(X_train_s)[:, 1]
p_test  = model.predict_proba(X_test_s)[:, 1]

auc_train = roc_auc_score(y_train, p_train)
auc_test  = roc_auc_score(y_test,  p_test)

print("train AUC:", round(auc_train, 4))
print("test  AUC:", round(auc_test, 4))
print("gap:      ", round(auc_train - auc_test, 4))

train AUC: 0.7156
test  AUC: 0.6975
gap:       0.0181
